In [11]:
df = pd.read_csv(r"C:\Users\nasib_vr6iaim\northstar-coursework\module-01-foundations-of-analytics-and-statistics\lesson-1-3-summary-statistics\skyline_enrollments.csv")

In [12]:
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

# ---- Find and load the data ----
matches = []
for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    matches += list(root.rglob("skyline_enrollments.csv"))

assert matches, (
    "Couldn't find skyline_enrollments.csv. Copy it into this notebook's folder "
    "or paste the full path into pd.read_csv(r'...')."
)
print("Using:", matches[0])
df = pd.read_csv(matches[0])

# ---- EDIT THESE ----
COURSE_A = "Python Basics"
COURSE_B = "Data Analytics"      # change to the course your assignment names
# --------------------

print("Courses in data:", df["course_name"].unique().tolist())
a = df.loc[df["course_name"] == COURSE_A, "hours_studied"].dropna()
b = df.loc[df["course_name"] == COURSE_B, "hours_studied"].dropna()
assert len(a) > 1 and len(b) > 1, "A course name didn't match. Compare to the list above."

n1, n2 = len(a), len(b)
m1, m2 = a.mean(), b.mean()
s1, s2 = a.std(ddof=1), b.std(ddof=1)
mean_diff = m1 - m2

# Welch standard error, degrees of freedom, and 95% CI for the difference
se_diff = np.sqrt(s1**2/n1 + s2**2/n2)
dof = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
t_crit = stats.t.ppf(0.975, dof)
ci = (mean_diff - t_crit*se_diff, mean_diff + t_crit*se_diff)

# Two-sample t-test (Welch)
t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)

# Effect size (Cohen's d)
sp = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1 + n2 - 2))
d = mean_diff / sp
size = "negligible" if abs(d) < 0.2 else "small" if abs(d) < 0.5 else "medium" if abs(d) < 0.8 else "large"

print(f"\n{COURSE_A}: n={n1}, mean={m1:.2f}, sd={s1:.2f}")
print(f"{COURSE_B}: n={n2}, mean={m2:.2f}, sd={s2:.2f}")
print(f"Mean difference ({COURSE_A} - {COURSE_B}): {mean_diff:.2f} hours")
print(f"95% CI for the difference: ({ci[0]:.2f}, {ci[1]:.2f}) hours")
print(f"p-value: {p_value:.4f}")
print(f"Cohen's d: {d:.2f} ({size} effect)")

Using: c:\Users\nasib_vr6iaim\northstar-coursework\module-01-foundations-of-analytics-and-statistics\lesson-1-3-summary-statistics\skyline_enrollments.csv
Courses in data: ['Python Basics', 'Data Analytics', 'Business Math', 'Intro to Statistics', 'Research Methods']

Python Basics: n=22, mean=11.69, sd=9.05
Data Analytics: n=25, mean=12.32, sd=8.42
Mean difference (Python Basics - Data Analytics): -0.63 hours
95% CI for the difference: (-5.79, 4.54) hours
p-value: 0.8072
Cohen's d: -0.07 (negligible effect)


Students in Python Basics studied about 0.6 hours less than students in Data Analytics on average (11.7 vs. 12.3 hours), but that gap is small enough that it could easily be due to chance. Our best estimate says the true difference could be anywhere from about 5.8 hours less to 4.5 hours more, and a result like this happens by chance about 8 times out of 10 even when there’s no real difference. A gap of under an hour also wouldn’t change anything in practice, given how much study time varies from student to student, so we can’t say these two courses differ in how much students study.


In [13]:
from itertools import combinations

rows = []
for a, b in combinations(sorted(df["course_name"].unique()), 2):
    x = df.loc[df["course_name"] == a, "hours_studied"]
    y = df.loc[df["course_name"] == b, "hours_studied"]
    t, p = stats.ttest_ind(x, y, equal_var=False)
    rows.append((a, b, round(x.mean() - y.mean(), 3), round(p, 4)))

results = pd.DataFrame(rows, columns=["course_a", "course_b", "mean_difference", "p_value"])
results.to_csv("pairwise_ttests.csv", index=False)

alpha_bonf = 0.05 / len(results)
print(results)
print("p < 0.05 (uncorrected):", (results["p_value"] < 0.05).sum())
print("Bonferroni threshold:", alpha_bonf)
print("Survive Bonferroni:", (results["p_value"] < alpha_bonf).sum())

              course_a             course_b  mean_difference  p_value
0        Business Math       Data Analytics            0.880   0.7111
1        Business Math  Intro to Statistics            3.386   0.1434
2        Business Math        Python Basics            1.509   0.5561
3        Business Math     Research Methods            6.200   0.0197
4       Data Analytics  Intro to Statistics            2.506   0.2798
5       Data Analytics        Python Basics            0.629   0.8072
6       Data Analytics     Research Methods            5.320   0.0404
7  Intro to Statistics        Python Basics           -1.877   0.4533
8  Intro to Statistics     Research Methods            2.814   0.2372
9        Python Basics     Research Methods            4.691   0.0827
p < 0.05 (uncorrected): 2
Bonferroni threshold: 0.005
Survive Bonferroni: 0


Two: Business Math vs. Research Methods (p = 0.020) and Data Analytics vs. Research Methods (p = 0.040).
Bonferroni-corrected threshold: 0.05 / 10 = 0.005.

What this says about the multiple testing trap: The courses were generated with no real differences, so every “significant” result here is sampling noise. Running 10 tests at 0.05 gives about a 40% chance of at least one false positive (1 − 0.95¹⁰), and we got two. Someone who reported only those two results would look like they found something real. Correcting for the number of tests, and reporting all 10, shows that nothing was found. Both false positives also involve Research Methods, which has only 5 students, a reminder that small samples produce unstable results.